In [30]:
import sys
from pathlib import Path

import pandas as pd

# get rid of warning
import warnings
warnings.filterwarnings("ignore")

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DIR, PROCESSED_DIR
from src.extract import (
    event_coverage_report,
    extract_all_events,
    get_epl_matches,
    save_raw,
)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

In [31]:
from statsbombpy import sb
from tqdm import tqdm

comps = sb.competitions()
# print(comps.head())

In [ ]:
results = []

for _, row in tqdm(comps.iterrows(), total=len(comps)):
    comp_id = row["competition_id"]
    season_id = row["season_id"]
    
    try:
        matches = sb.matches(competition_id=comp_id, season_id=season_id)
        match_count = len(matches)

        event_count = 0

        # sample a few matches to estimate event density (faster than full scan)
        sample_matches = matches["match_id"].head(min(5, len(matches)))

        for mid in sample_matches:
            events = sb.events(match_id=mid)
            event_count += len(events)

        avg_events_per_match = event_count / len(sample_matches) if len(sample_matches) > 0 else 0
        est_total_events = avg_events_per_match * match_count

        results.append({
            "competition": row["competition_name"],
            "country": row["country_name"],
            "season": season_id,
            "matches": match_count,
            "avg_events_per_match": avg_events_per_match,
            "estimated_total_events": est_total_events
        })

    except Exception as e:
        print(f"Skipped {comp_id}-{season_id}: {e}")

df = pd.DataFrame(results)

### Survey note

`estimated_total_events` is a **projection only**: it samples **5 matches** per competition-season, computes `avg_events_per_match`, then multiplies by total match count. It does **not** download or save events. Actual volume comes from the extraction cells below.

In [33]:
df.sort_values("estimated_total_events", ascending=False).head(10)

,competition,country,season,matches,avg_events_per_match,estimated_total_events
60,Ligue 1,France,27,377,3872.6,1459970.2
66,Serie A,Italy,27,380,3685.2,1400376.0
64,Premier League,England,27,380,3404.0,1293520.0
43,La Liga,Spain,27,380,3044.2,1156796.0
1,1. Bundesliga,Germany,27,306,3560.6,1089543.6
25,FA Women's Super League,England,90,131,3587.2,469923.2
27,FA Women's Super League,England,4,108,3334.6,360136.8
37,Indian Super league,India,108,115,3090.4,355396.0
26,FA Women's Super League,England,42,87,3039.8,264462.6
29,FIFA World Cup,International,106,64,3594.2,230028.8


## Premier League extraction

Goal: analyze playstyle change (2003/04 vs 2015/16). The survey above ranked competitions; this section downloads **all** EPL match events via `src.extract`.

In [34]:
matches_df = get_epl_matches()
print(f"EPL matches cataloged: {len(matches_df)} ({matches_df['season'].nunique()} seasons)")
matches_df.groupby("season").size()

EPL matches cataloged: 418 (2 seasons)


season
2003/2004     38
2015/2016    380
dtype: int64

In [35]:
# resume=True skips match_ids already saved under data/raw/events_parts/
events_df = extract_all_events(matches_df, RAW_DIR, resume=True)
print(f"Extracted {len(events_df):,} events from {events_df['match_id'].nunique()} matches")

Resuming: skipping 418 matches already on disk, 0 remaining.


Extracting events: 0it [00:00, ?it/s]


Extracted 1,442,679 events from 418 matches


In [36]:
# print(events_df.head())
print(events_df.columns.tolist())

['bad_behaviour_card', 'ball_receipt_outcome', 'ball_recovery_offensive', 'ball_recovery_recovery_failure', 'block_offensive', 'carry_end_location', 'clearance_aerial_won', 'clearance_body_part', 'clearance_head', 'clearance_left_foot', 'clearance_right_foot', 'counterpress', 'dribble_nutmeg', 'dribble_outcome', 'dribble_overrun', 'duel_outcome', 'duel_type', 'duration', 'foul_committed_advantage', 'foul_committed_card', 'foul_committed_offensive', 'foul_committed_type', 'foul_won_advantage', 'foul_won_defensive', 'goalkeeper_body_part', 'goalkeeper_end_location', 'goalkeeper_outcome', 'goalkeeper_position', 'goalkeeper_technique', 'goalkeeper_type', 'id', 'index', 'interception_outcome', 'location', 'match_id', 'minute', 'miscontrol_aerial_won', 'off_camera', 'out', 'pass_aerial_won', 'pass_angle', 'pass_assisted_shot_id', 'pass_body_part', 'pass_cross', 'pass_cut_back', 'pass_deflected', 'pass_end_location', 'pass_goal_assist', 'pass_height', 'pass_inswinging', 'pass_length', 'pass_m

In [37]:
save_raw(matches_df, events_df, RAW_DIR)

coverage = event_coverage_report(matches_df, events_df)
print(f"Total events saved: {len(events_df):,}\n")
print("Matches vs events captured by season:")
print(coverage)

# Compare 2015/16 to survey projection (season_id 27 in survey df)
epl_survey = df[(df["competition"] == "Premier League") & (df["season"] == 27)]
if not epl_survey.empty:
    est = epl_survey["estimated_total_events"].iloc[0]
    actual = coverage.loc["2015/2016", "events"] if "2015/2016" in coverage.index else 0
    print(f"\n2015/16 survey estimate: {est:,.0f} | actual events: {actual:,}")

Total events saved: 1,442,679

Matches vs events captured by season:
           matches  matches_with_events   events  missing_matches
season                                                           
2003/2004       38                   38   129401                0
2015/2016      380                  380  1313278                0

2015/16 survey estimate: 1,293,520 | actual events: 1,313,278
